In [ ]:
# ============================================================
# 🚀 EDA COMPACTO Y OPTIMIZADO (DATOS SECOP)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
import unicodedata
import os

# ============================================================
# 1. CARGA DE ARCHIVOS
# ============================================================

archivos = {
    "categoria":      "D_Categoria.xlsx",
    "entidad":        "D_Entidad.xlsx",
    "modalidad":      "D_Modalidad.xlsx",
    "proveedor":      "D_Proveedor.xlsx",
    "tiempo":         "D_Tiempo.xlsx",
    "tipocontrato":   "D_TipoContrato.xlsx",
    "ubientidad":     "D_UbiEntidad.xlsx",
    "ubiproveedor":   "D_UbiProveedor.xlsx",
    "fact_1":         "F_Proceso_parte1.xlsx",
    "fact_2":         "F_Proceso_parte2.xlsx"
}

dfs = {k: pd.read_excel(v) for k,v in archivos.items()}
df = pd.concat([dfs["fact_1"], dfs["fact_2"]], ignore_index=True)

print("✔ Archivos cargados y tabla FACT unificada.")


# ============================================================
# 2. LIMPIEZA GENERAL
# ============================================================

def normalizar(x):
    if pd.isna(x): return np.nan
    x = str(x).strip().upper()
    x = unicodedata.normalize('NFKD', x).encode('ascii','ignore').decode()
    if x in ["", "NA", "N/A", "-", "--", "NULL", "SIN DATOS"]: return np.nan
    return x

cat_cols = df.select_dtypes(include=["object","category"]).columns
num_cols = df.select_dtypes(include=[np.number]).columns

for c in cat_cols:
    df[c] = df[c].apply(normalizar)

print("✔ Variables categóricas limpiadas.")


# ============================================================
# 3. DATOS FALTANTES
# ============================================================

faltantes = df.isnull().sum().sort_values(ascending=False).to_frame("Nulos")
faltantes["%"] = (df.isnull().mean()*100).round(2)

print("📌 Variables con más datos faltantes:")
display(faltantes.head(10))


# ============================================================
# 4. IMPUTACIÓN AUTOMÁTICA
# ============================================================

df[num_cols] = SimpleImputer(strategy="median").fit_transform(df[num_cols])
df[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(df[cat_cols])

print("✔ Imputación aplicada (mediana numérica, moda categórica).")


# ============================================================
# 5. OUTLIERS (IQR)
# ============================================================

outlier_resumen = {}

for col in num_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    low, high = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outlier_resumen[col] = df[(df[col] < low) | (df[col] > high)][col].count()

outlier_df = pd.DataFrame.from_dict(outlier_resumen, orient="index", columns=["Outliers"])
print("📌 Outliers detectados por variable:")
display(outlier_df.sort_values("Outliers", ascending=False))


# ============================================================
# 6. ANÁLISIS UNIVARIADO
# ============================================================

print("\n📊 Estadísticos numéricos:")
display(df[num_cols].describe().T)

print("\n📋 Frecuencias categóricas (Top 10):")
for col in cat_cols:
    print(f"\n🔹 {col}")
    display(df[col].value_counts().head(10))


# ============================================================
# 7. VISUALIZACIONES (AUTO)
# ============================================================

# Histogramas
for col in num_cols:
    plt.figure(figsize=(5,3))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribución: {col}")
    plt.show()

# Pareto categórico
for col in cat_cols:
    plt.figure(figsize=(6,3))
    df[col].value_counts().head(10).plot(kind="bar")
    plt.title(f"Top 10 categorías: {col}")
    plt.tight_layout()
    plt.show()


# ============================================================
# 8. EXPORTACIÓN
# ============================================================

faltantes.to_excel("reporte_faltantes.xlsx")
outlier_df.to_excel("reporte_outliers.xlsx")
df.describe().to_excel("reporte_descriptivo.xlsx")

print("✔ Reportes exportados a Excel.")


# ============================================================
# 9. REPORTE FINAL
# ============================================================

print("===================================================")
print("📘 REPORTE FINAL DEL EDA")
print("===================================================")
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])
print("Variables numéricas:", len(num_cols))
print("Variables categóricas:", len(cat_cols))
print("Columnas con datos faltantes antes de imputar:")
display(faltantes.head())

print("\nProceso completado ✔✔✔")
